In [1]:
import numpy as np
import pandas as pd

In [2]:
embs = pd.read_parquet('embeddings_policies_Qwen3-4B_2026-03-05.parquet')

In [3]:
df = pd.read_parquet('clustered_policies_varying_gamma_2026-03-09.parquet', columns=['openalex_id', 'chunk_idx', 'policy_text', 'cluster_id_gamma_150'])

In [5]:
embs

,openalex_id,chunk_idx,domain,policy_type,actor,policy_text,embedding
0,W1000416519,0,URBAN,ECONOMIC,NATIONAL,Buyouts of property and structures as part of ...,"[-0.000351, 0.04614, 0.02902, 0.01041, -0.0021..."
1,W1000416519,0,URBAN,ECONOMIC,REGIONAL,Prioritization of parcels for buyouts to reduc...,"[-0.0004473, 0.03687, 0.008736, 0.01563, -0.00..."
2,W1000416519,0,NATURE,SPATIAL,NATIONAL,"Restoration of natural resources, including we...","[-0.0002345, -0.00078, -0.02588, -0.003849, -0..."
3,W1000416519,0,NATURE,SPATIAL,REGIONAL,Restoration of salmonid habitat in Sonoma Coun...,"[-0.0002744, 0.05298, -0.01653, 0.00801, -0.00..."
4,W1000416519,1,NATURE,ECONOMIC,NATIONAL,National Oceanic and Atmospheric Administratio...,"[-0.0001951, 0.02393, -0.00818, 0.06274, -0.00..."
...,...,...,...,...,...,...,...
1466022,W991832166,5,SOCIAL,REGULATORY,NATIONAL,Minority lists that fail to win preferential m...,"[-0.000576, 0.01172, -0.01033, 0.008766, -0.00..."
1466023,W991832166,5,SOCIAL,REGULATORY,NATIONAL,Minority MPs and spokespersons can found perma...,"[-0.0006123, 0.01182, -0.01665, 0.0316, -0.001..."
1466024,W991832166,5,SOCIAL,REGULATORY,NATIONAL,The new constitution places lesser importance ...,"[-0.0005507, 0.06198, 0.0279, 0.0116, -0.00221..."
1466025,W991832166,5,SOCIAL,REGULATORY,NATIONAL,The general line of internal minority policy h...,"[-0.0004122, 0.06134, 0.0211, 0.03714, -0.0014..."


In [4]:
df

,openalex_id,chunk_idx,policy_text,cluster_id_gamma_150
0,W1000416519,0,Buyouts of property and structures as part of ...,0
1,W1000416519,0,Prioritization of parcels for buyouts to reduc...,0
2,W1000416519,0,"Restoration of natural resources, including we...",1
3,W1000416519,0,Restoration of salmonid habitat in Sonoma Coun...,2
4,W1000416519,1,National Oceanic and Atmospheric Administratio...,3
...,...,...,...,...
1466022,W991832166,5,Minority lists that fail to win preferential m...,684
1466023,W991832166,5,Minority MPs and spokespersons can found perma...,684
1466024,W991832166,5,The new constitution places lesser importance ...,684
1466025,W991832166,5,The general line of internal minority policy h...,684


In [6]:
df['embedding'] = embs['embedding']

In [8]:
df = df.rename(columns={'policy_text': 'text', 'cluster_id_gamma_150': 'cluster_id'})

In [11]:
means = df.groupby('cluster_id')['embedding'].mean()

In [12]:
means

cluster_id
0       [-0.0002910907451923077, 0.03013548951048951, ...
1       [-0.000250694735738255, -0.004041701621923938,...
2       [-0.00030167897542317707, 0.0294189453125, -0....
3       [-0.00038320806962025315, 0.04956981803797468,...
4       [-0.0002656894095538312, -0.002339582322987391...
                              ...                        
3794    [-0.00028252601623535156, 0.0220184326171875, ...
3795    [-0.00013959407806396484, 0.0275421142578125, ...
3796    [-0.00040531158447265625, 0.057830810546875, -...
3797    [-0.00023102760314941406, 0.0168609619140625, ...
3798    [-0.0002696514129638672, 0.05120849609375, -0....
Name: embedding, Length: 3799, dtype: object

In [31]:
np.linalg.norm(means.iloc[0])

np.float64(0.7938511000469534)

In [36]:
nmeans = means / means.apply(np.linalg.norm)

## Delete small clusters
We don't reassign policies from deleted clusters to the cluster with the closest mean, as in many cases it's still very distant.

In [13]:
sizes = df['cluster_id'].value_counts()

In [16]:
small_cluster_ids = sizes[sizes <= 2].index.values

In [41]:
orphans = df[df['cluster_id'].isin(small_cluster_ids)]

In [43]:
orphemb = np.stack(orphans['embedding'])

In [48]:
npmeans = np.stack(nmeans)

In [83]:
sims = orphemb @ npmeans.T
sims.shape

(1330, 3799)

In [106]:
np.sort(sims, axis=1)[:, -2].mean() # there is usually no close cluster

np.float64(0.6676524328809926)

In [111]:
df = df.drop(orphans.index)

In [139]:
df.cluster_id.nunique()

2625

## Computing medoids

In [115]:
from tqdm.auto import tqdm

In [131]:
representatives = {}
for cluster_id, group in tqdm(df.groupby('cluster_id')):
    mu = nmeans.loc[cluster_id]
    embs = np.stack(group['embedding'])
    sims = embs @ mu
    closest_id = sims.argmax()
    closest = group.iloc[closest_id]
    representatives[cluster_id] = closest.name

  0%|          | 0/2625 [00:00<?, ?it/s]

In [140]:
df['representative'] = False

In [143]:
df.loc[representatives.values(), 'representative'] = True

In [146]:
df[df['representative'] == True]

,openalex_id,chunk_idx,text,cluster_id,embedding,representative
1232,W109263537,2,Involvement of stakeholders in the land-use pl...,587,"[-0.0002354, 0.01487, -0.02782, -0.00477, -0.0...",True
3132,W121033267,0,Integration of restorative justice approach an...,743,"[-0.0002925, 0.0415, -0.02556, 0.01381, -0.001...",True
3223,W121420035,2,"Recommendation for model refinement, particula...",1069,"[-0.000286, 0.0314, -0.02524, -0.02519, -0.001...",True
4366,W130298104,0,Dissemination of research findings to policy m...,1035,"[-0.0002059, 0.0461, -0.05304, 0.02115, -0.001...",True
4852,W135063704,1,Research on the effects of organic and convent...,1272,"[-0.0003061, 0.05875, -0.06082, -0.05737, -0.0...",True
...,...,...,...,...,...,...
1464148,W7104666906,2,Strengthen community engagement and empowermen...,2073,"[-0.0004404, 0.02258, -0.04062, -0.01403, -0.0...",True
1464187,W7104730991,2,Textile Strategy to promote sustainable practi...,393,"[-0.0003934, 0.03876, 0.00767, -0.01634, -0.00...",True
1464280,W7105117417,0,Promotion of climate-smart agricultural practi...,311,"[-0.000374, 0.04196, -0.04196, -0.04333, -0.00...",True
1464345,W7105603035,0,Government support for health-promoting EBIs f...,2231,"[-0.0004478, 0.0713, -0.03424, 0.0403, -0.0026...",True


In [149]:
df.reset_index(drop=True)

,openalex_id,chunk_idx,text,cluster_id,embedding,representative
0,W1000416519,0,Buyouts of property and structures as part of ...,0,"[-0.000351, 0.04614, 0.02902, 0.01041, -0.0021...",False
1,W1000416519,0,Prioritization of parcels for buyouts to reduc...,0,"[-0.0004473, 0.03687, 0.008736, 0.01563, -0.00...",False
2,W1000416519,0,"Restoration of natural resources, including we...",1,"[-0.0002345, -0.00078, -0.02588, -0.003849, -0...",False
3,W1000416519,0,Restoration of salmonid habitat in Sonoma Coun...,2,"[-0.0002744, 0.05298, -0.01653, 0.00801, -0.00...",False
4,W1000416519,1,National Oceanic and Atmospheric Administratio...,3,"[-0.0001951, 0.02393, -0.00818, 0.06274, -0.00...",False
...,...,...,...,...,...,...
1464692,W991832166,5,Minority lists that fail to win preferential m...,684,"[-0.000576, 0.01172, -0.01033, 0.008766, -0.00...",False
1464693,W991832166,5,Minority MPs and spokespersons can found perma...,684,"[-0.0006123, 0.01182, -0.01665, 0.0316, -0.001...",False
1464694,W991832166,5,The new constitution places lesser importance ...,684,"[-0.0005507, 0.06198, 0.0279, 0.0116, -0.00221...",False
1464695,W991832166,5,The general line of internal minority policy h...,684,"[-0.0004122, 0.06134, 0.0211, 0.03714, -0.0014...",False


In [153]:
df = df.drop(columns='embedding')

In [155]:
df.reset_index(drop=True).to_parquet('clustered_policies_with_representatives_2026-03-10.parquet')

In [160]:
sizes

cluster_id
665     3223
1518    2897
21      2882
915     2849
839     2740
        ... 
3794       1
3795       1
3796       1
3797       1
3798       1
Name: count, Length: 3799, dtype: int64

In [165]:
clust = df.loc[representatives.values()].merge(sizes, on='cluster_id')[['cluster_id', 'text', 'count']]
clust.to_csv('cluster_representatives_2026-03-10.csv', index=False)

In [121]:
a = 0
for cluster_id, rep in representatives.items():
    print(rep['text'])
    s = 0
    for i, row in df[df['cluster_id'] == cluster_id].iterrows():
        print('\t', row['text'])
        if s >= 10:
            break
        s += 1
    print('-'*40)
    if a >= 100:
        break
    a += 1

Insurance programs and tax incentives for flood risk
	 Buyouts of property and structures as part of FEMA’s overall risk reduction strategy
	 Prioritization of parcels for buyouts to reduce long-term risks and efficiently use limited governmental funds
	 FEMA’s Hazard Mitigation Grant Programs with obligated funds exceeding $700 million in 2013
	 Insurers should work with municipal governments and civil society to take an active role in preventing unnecessary losses by becoming involved in urban planning decisions.
	 Insurers could lobby for stricter provincial building codes for municipalities to reduce or prevent loss.
	 Insurers can educate consumer markets about the growing climate risk and provide information about which measures protect against storm damage.
	 Collaborating with government to protect against catastrophic losses.
	 Insurers could lobby the federal government to institute a government insurance program to re-insure the growing costs of catastrophic wildfires within

In [129]:
repdf = pd.DataFrame(representatives).T
repdf

,openalex_id,chunk_idx,text,cluster_id,embedding
0,W3034226054,0,Insurance programs and tax incentives for floo...,0,"[-0.0002165, 0.03012, 0.01015, 0.01595, -0.001..."
1,W3137623305,0,Preserving natural lands for flood management,1,"[-0.000114, 0.011856, -0.02217, 0.008194, -0.0..."
2,W4224955221,1,Continuous monitoring of fishways to optimise ...,2,"[-0.0002165, 0.02953, -0.05716, 0.006554, -0.0..."
3,W4388022516,3,Increasing restoration funding,3,"[-0.000325, 0.05966, 0.00443, 0.01685, -0.0019..."
4,W2587122776,0,Implementation of measures to mitigate the ris...,4,"[-0.0001707, -0.01356, -0.02719, -0.01584, -0...."
...,...,...,...,...,...
3673,W4412189162,0,Normativas de gestión de riesgos hídricos para...,3673,"[-0.0001266, 0.00054, -0.01688, 0.00635, -0.00..."
3702,W4413790643,7,Investments in infrastructure directly address...,3702,"[-0.0003774, 0.02861, -0.01226, -0.01043, -0.0..."
3708,W4413291661,0,Target Environment-Specific Deterioration Mana...,3708,"[-0.000606, 0.03757, -0.00348, -0.04715, -0.00..."
3710,W4414896874,0,Equity-led policies,3710,"[-0.0002997, 0.05832, -0.00622, 0.03204, -0.00..."


In [130]:
df.loc[repdf.index]

KeyError: '[175, 558, 1139, 2211] not in index'